# Consistency in L features

In [1]:
import os
import re
import glob
from typing import List, Optional, Tuple, Dict, Literal

import numpy as np
import pandas as pd
from scipy import stats

# -------------------------
# Generic directory loader (summary_sc_<id>.csv OR turns_<id>.csv)
# -------------------------
def load_participant_matrix_from_dir(
    data_dir: str,
    *,
    pattern: str,
    id_regex: str,
    features: Optional[List[str]] = None,
    # for one-row files use "first"; for turns_<id>.csv use "median" (recommended)
    row_agg: Literal["first", "mean", "median"] = "first",
    # participant-only turns filter (best-effort heuristic)
    participant_only: bool = True,
    speaker_col_candidates: Tuple[str, ...] = (
        "speaker", "Speaker", "role", "Role", "speaker_role", "SpeakerRole", "who", "Who"
    ),
    # values that indicate interviewer/Ellie to exclude
    exclude_speakers_substrings: Tuple[str, ...] = ("ellie", "interviewer", "assistant"),
) -> pd.DataFrame:
    """
    Reads many CSVs like summary_sc_<id>.csv or turns_<id>.csv into one DataFrame:
      index = Participant (id from filename)
      columns = aggregated features

    For turns_<id>.csv (multiple rows), set row_agg="median" (default recommendation).
    """
    if not os.path.isdir(data_dir):
        raise FileNotFoundError(f"Not a directory: {data_dir}")

    files = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"No files found: {os.path.join(data_dir, pattern)}")

    rx = re.compile(id_regex)
    rows = []
    for fp in files:
        m = rx.search(os.path.basename(fp))
        if not m:
            continue
        pid = int(m.group(1))

        df = pd.read_csv(fp)
        if df.empty:
            continue

        # optional: keep participant-only turns by removing Ellie/interviewer rows
        if participant_only:
            spk_col = next((c for c in speaker_col_candidates if c in df.columns), None)
            if spk_col is not None:
                spk = df[spk_col].astype(str).str.lower()
                mask = np.ones(len(df), dtype=bool)
                for bad in exclude_speakers_substrings:
                    mask &= ~spk.str.contains(bad, na=False)
                df = df.loc[mask]
                if df.empty:
                    continue

        # keep only requested features (+ any needed for filtering already done)
        if features is not None:
            keep = [c for c in features if c in df.columns]
            df = df[keep].copy()

        # convert columns to numeric where possible
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        if df.empty:
            continue

        if row_agg == "first":
            rec = df.iloc[0].to_dict()
        elif row_agg == "mean":
            rec = df.mean(axis=0, numeric_only=True).to_dict()
        elif row_agg == "median":
            rec = df.median(axis=0, numeric_only=True).to_dict()
        else:
            raise ValueError("row_agg must be one of: 'first','mean','median'")

        rec["Participant"] = pid
        rows.append(rec)

    if not rows:
        raise ValueError(f"Parsed 0 valid files in: {data_dir}")

    out = pd.DataFrame(rows).set_index("Participant").sort_index()
    return out


# -------------------------
# Agreement metrics (EN vs UA)
# -------------------------
def _paired_clean(a: np.ndarray, b: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    m = np.isfinite(a) & np.isfinite(b)
    return a[m], b[m]


def _bootstrap_ci(values_fn, a: np.ndarray, b: np.ndarray, n_boot: int = 2000, seed: int = 1706) -> Tuple[float, float, float]:
    rng = np.random.default_rng(seed)
    a, b = _paired_clean(a, b)
    n = len(a)
    if n < 3:
        return np.nan, np.nan, np.nan

    stats_list = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        s = values_fn(a[idx], b[idx])
        if np.isfinite(s):
            stats_list.append(s)

    if not stats_list:
        return np.nan, np.nan, np.nan

    lo, hi = np.percentile(stats_list, [2.5, 97.5])
    return float(np.mean(stats_list)), float(lo), float(hi)


def _pearson(a: np.ndarray, b: np.ndarray) -> float:
    a, b = _paired_clean(a, b)
    if len(a) < 3 or np.allclose(a, a[0]) or np.allclose(b, b[0]):
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def _spearman(a: np.ndarray, b: np.ndarray) -> float:
    a, b = _paired_clean(a, b)
    if len(a) < 3:
        return np.nan
    r, _ = stats.spearmanr(a, b, nan_policy="omit")
    return float(r)


def icc2_1(a: np.ndarray, b: np.ndarray) -> float:
    """
    ICC(2,1): two-way random effects, absolute agreement, single measurement.
    EN and UA are the two "raters"; subjects are participants.
    """
    a, b = _paired_clean(a, b)
    n = len(a)
    k = 2
    if n < 3:
        return np.nan

    Y = np.vstack([a, b]).T  # (n, 2)
    mean_subject = Y.mean(axis=1, keepdims=True)
    mean_rater = Y.mean(axis=0, keepdims=True)
    grand_mean = Y.mean()

    SSR = k * np.sum((mean_subject - grand_mean) ** 2)
    SSC = n * np.sum((mean_rater - grand_mean) ** 2)
    SSE = np.sum((Y - mean_subject - mean_rater + grand_mean) ** 2)

    MSR = SSR / (n - 1)
    MSC = SSC / (k - 1)
    MSE = SSE / ((n - 1) * (k - 1))

    denom = MSR + (k - 1) * MSE + (k * (MSC - MSE) / n)
    if denom <= 0:
        return np.nan
    return float((MSR - MSE) / denom)


def _mean_shift(a: np.ndarray, b: np.ndarray) -> float:
    a, b = _paired_clean(a, b)
    if len(a) == 0:
        return np.nan
    return float(np.mean(b - a))  # UA - EN


def _median_min_max(x: np.ndarray) -> Tuple[float, float, float]:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    return float(np.median(x)), float(np.min(x)), float(np.max(x))


# -------------------------
# Main API (works for summary_sc_*.csv AND turns_*.csv)
# -------------------------
def check_statistic(
    *,
    features: List[str],
    dataset_eng_dir: str,
    dataset_ukr_dir: str,
    file_kind: Literal["summary", "turns"] = "summary",
    n_boot: int = 2000,
    seed: int = 1706,
    strict: bool = False,
    # turns only: how to aggregate multiple turn rows -> one value per participant per feature
    turns_agg: Literal["mean", "median"] = "median",
    # turns only: try to exclude interviewer/Ellie rows
    participant_only_turns: bool = True,
) -> pd.DataFrame:
    """
    Computes EN vs UA agreement per feature:
      Pearson r, Spearman rho, ICC(2,1), mean shift Δμ = mean(UA-EN), with bootstrap 95% CI.

    If file_kind="turns", we first aggregate each turns_<id>.csv into a single row per participant
    using turns_agg (default median). This matches your “turn-level -> interview-level” aggregation logic.
    """
    if file_kind == "summary":
        pattern = "summary_sc_*.csv"
        id_regex = r"summary_sc_(\d+)\.csv$"
        row_agg = "first"
        participant_only = False  # summary is already participant-only
    elif file_kind == "turns":
        pattern = "turns_*.csv"
        id_regex = r"turns_(\d+)\.csv$"
        row_agg = turns_agg
        participant_only = participant_only_turns
    else:
        raise ValueError("file_kind must be 'summary' or 'turns'")

    df_en = load_participant_matrix_from_dir(
        dataset_eng_dir,
        pattern=pattern,
        id_regex=id_regex,
        features=features,
        row_agg=row_agg,
        participant_only=participant_only,
    )
    df_ua = load_participant_matrix_from_dir(
        dataset_ukr_dir,
        pattern=pattern,
        id_regex=id_regex,
        features=features,
        row_agg=row_agg,
        participant_only=participant_only,
    )

    merged = df_en.add_suffix("_EN").join(df_ua.add_suffix("_UA"), how="inner")
    if merged.empty:
        raise ValueError("No overlapping participants between EN and UA dirs.")

    rows: List[Dict[str, object]] = []

    for feat in features:
        c_en = f"{feat}_EN"
        c_ua = f"{feat}_UA"

        if c_en not in merged.columns or c_ua not in merged.columns:
            if strict:
                raise KeyError(f"Missing feature in merged data: {feat}")
            rows.append({
                "feature": feat,
                "n": 0,
                "EN_median[min,max]": "",
                "UA_median[min,max]": "",
                "r [95% CI]": "",
                "rho [95% CI]": "",
                "ICC(2,1) [95% CI]": "",
                "mean shift Δμ (UA-EN) [95% CI]": "",
                "note": "missing in EN/UA files",
            })
            continue

        a = merged[c_en].to_numpy(dtype=float)  # EN
        b = merged[c_ua].to_numpy(dtype=float)  # UA
        a, b = _paired_clean(a, b)
        n = len(a)

        en_med, en_min, en_max = _median_min_max(a)
        ua_med, ua_min, ua_max = _median_min_max(b)

        r_m, r_lo, r_hi = _bootstrap_ci(_pearson, a, b, n_boot=n_boot, seed=seed)
        s_m, s_lo, s_hi = _bootstrap_ci(_spearman, a, b, n_boot=n_boot, seed=seed)

        icc_mean, icc_lo, icc_hi = _bootstrap_ci(lambda x, y: icc2_1(x, y), a, b, n_boot=n_boot, seed=seed)
        sh_m, sh_lo, sh_hi = _bootstrap_ci(_mean_shift, a, b, n_boot=n_boot, seed=seed)

        rows.append({
            "feature": feat,
            "n": int(n),
            "EN_median[min,max]": f"{en_med:.3f} [{en_min:.3f},{en_max:.3f}]",
            "UA_median[min,max]": f"{ua_med:.3f} [{ua_min:.3f},{ua_max:.3f}]",
            "r [95% CI]": f"{r_m:.3f} [{r_lo:.3f},{r_hi:.3f}]",
            "rho [95% CI]": f"{s_m:.3f} [{s_lo:.3f},{s_hi:.3f}]",
            "ICC(2,1) [95% CI]": f"{icc_mean:.3f} [{icc_lo:.3f},{icc_hi:.3f}]",
            "mean shift Δμ (UA-EN) [95% CI]": f"{sh_m:.3f} [{sh_lo:.3f},{sh_hi:.3f}]",
            "note": "",
        })

    out = pd.DataFrame(rows).sort_values(["n", "feature"], ascending=[False, True]).reset_index(drop=True)
    return out

# -------------------------
# Example usage
# -------------------------
L_SUMMARY_FEATURES = [
    "syllables_per_min",
    "sentiment_pos",
    "sentiment_neg",
    "sentiment_neu",
    "sentiment_overall",
    "mattr_5",
    "mattr_10",
    "mattr_25",
    "mattr_50",
    "mattr_100",
    "first_person_percentage",
    "first_person_sentiment_positive",
    "first_person_sentiment_negative",
    "first_person_sentiment_overall",
    "word_coherence_mean",
    "word_coherence_var",
    "word_coherence_5_mean",
    "word_coherence_5_var",
    "word_coherence_10_mean",
    "word_coherence_10_var",
    *[f"word_coherence_variability_{k}_mean" for k in range(2, 11)],
    *[f"word_coherence_variability_{k}_var" for k in range(2, 11)],
    "first_order_sentence_tangeniality_mean",
    "first_order_sentence_tangeniality_var",
    "second_order_sentence_tangeniality_mean",
    "second_order_sentence_tangeniality_var",
    "turn_to_turn_tangeniality_mean",
    "turn_to_turn_tangeniality_var",
    "turn_to_turn_tangeniality_slope",
    "semantic_perplexity_mean",
    "semantic_perplexity_var",
    "semantic_perplexity_5_mean",
    "semantic_perplexity_5_var",
    "semantic_perplexity_11_mean",
    "semantic_perplexity_11_var",
    "semantic_perplexity_15_mean",
    "semantic_perplexity_15_var",
]


In [3]:
stats_df = check_statistic(
    features=L_SUMMARY_FEATURES,
    dataset_eng_dir="/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_eng_norm_gemma",
    dataset_ukr_dir="/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_ukr_norm_gemma",
    file_kind="summary",
    n_boot=2000,
    seed=1706,
)
stats_df.drop(columns=["note", 'n'], inplace=True)
stats_df

,feature,"EN_median[min,max]","UA_median[min,max]",r [95% CI],rho [95% CI],"ICC(2,1) [95% CI]",mean shift Δμ (UA-EN) [95% CI]
0,first_order_sentence_tangeniality_mean,"0.713 [0.583,0.869]","0.700 [0.557,0.854]","0.977 [0.972,0.982]","0.973 [0.964,0.980]","0.947 [0.935,0.957]","-0.013 [-0.014,-0.012]"
1,second_order_sentence_tangeniality_mean,"0.702 [0.584,0.866]","0.696 [0.556,0.854]","0.977 [0.972,0.982]","0.975 [0.968,0.981]","0.960 [0.951,0.968]","-0.010 [-0.011,-0.009]"
2,semantic_perplexity_11_mean,"1784.864 [625.062,5827.228]","11294.125 [26.730,40276.762]","0.509 [0.390,0.612]","0.516 [0.414,0.605]","0.032 [0.022,0.043]","10265.147 [9678.693,10866.292]"
3,semantic_perplexity_15_mean,"1261.687 [475.882,4628.324]","5538.000 [18.433,18335.338]","0.598 [0.502,0.678]","0.594 [0.498,0.674]","0.075 [0.055,0.096]","4589.509 [4298.794,4890.470]"
4,semantic_perplexity_5_mean,"9250.801 [4809.699,29597.453]","186928.969 [32.228,827845.812]","0.375 [0.249,0.501]","0.454 [0.349,0.553]","0.004 [0.003,0.006]","187268.058 [177610.070,197607.723]"
5,semantic_perplexity_mean,"35.462 [16.069,138.998]","25.537 [1.486,85.538]","0.689 [0.593,0.762]","0.718 [0.638,0.792]","0.429 [0.353,0.501]","-14.766 [-16.535,-13.091]"
6,syllables_per_min,"117.076 [28.693,251.182]","139.042 [32.973,777.020]","0.779 [0.720,0.839]","0.930 [0.884,0.967]","0.459 [0.384,0.545]","45.918 [37.484,55.169]"
7,first_person_percentage,"2.447 [0.447,5.763]","8.363 [1.160,24.390]","0.201 [0.088,0.323]","0.267 [0.155,0.379]","0.019 [0.008,0.031]","5.809 [5.537,6.102]"
8,first_person_sentiment_negative,"0.171 [0.042,0.749]","4.563 [0.275,10.142]","0.272 [0.161,0.383]","0.300 [0.191,0.404]","0.004 [0.002,0.006]","4.304 [4.127,4.479]"
9,first_person_sentiment_overall,"18.540 [0.454,31.237]","4.601 [0.655,39.785]","-0.121 [-0.207,-0.025]","-0.107 [-0.227,0.016]","-0.016 [-0.030,-0.003]","-13.725 [-14.402,-13.042]"


In [ ]:
stats_df.to_csv("/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/agreement_summary_EN_vs_UA.csv", index=False)

In [4]:
L_TURNS = [
    "syllables_per_min",
    "sentiment_pos",
    "sentiment_neg",
    "sentiment_neu",
    "sentiment_overall",
    "mattr_5",
    "mattr_10",
    "mattr_25",
    "mattr_50",
    "mattr_100",
    "first_person_percentage",
    "first_person_sentiment_positive",
    "first_person_sentiment_negative",
    # discourse (per-turn)
    "first_order_sentence_tangeniality",
    "second_order_sentence_tangeniality",
    "turn_to_turn_tangeniality",
    "semantic_perplexity",
    "semantic_perplexity_5",
    "semantic_perplexity_11",
    "semantic_perplexity_15",
]

# 2) Turns files (turns_<pid>.csv), aggregated per participant by median:
stats_turns = check_statistic(
    features=L_TURNS,  # your per-turn feature list
    dataset_eng_dir="/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_eng_norm_gemma",
    dataset_ukr_dir="/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_ukr_norm_gemma",
    file_kind="turns",
    turns_agg="median",
    # participant_only=True,
)

stats_turns.drop(columns=["note", 'n'], inplace=True)
stats_turns

,feature,"EN_median[min,max]","UA_median[min,max]",r [95% CI],rho [95% CI],"ICC(2,1) [95% CI]",mean shift Δμ (UA-EN) [95% CI]
0,first_order_sentence_tangeniality,"0.713 [0.583,0.869]","0.700 [0.557,0.854]","0.977 [0.972,0.982]","0.973 [0.964,0.980]","0.947 [0.935,0.957]","-0.013 [-0.014,-0.012]"
1,second_order_sentence_tangeniality,"0.702 [0.584,0.866]","0.696 [0.556,0.854]","0.977 [0.972,0.982]","0.975 [0.968,0.981]","0.960 [0.951,0.968]","-0.010 [-0.011,-0.009]"
2,semantic_perplexity,"35.462 [16.069,138.998]","25.537 [1.486,85.538]","0.689 [0.593,0.762]","0.718 [0.638,0.792]","0.429 [0.353,0.501]","-14.766 [-16.535,-13.091]"
3,semantic_perplexity_11,"1784.864 [625.062,5827.228]","11294.125 [26.730,40276.762]","0.509 [0.390,0.612]","0.516 [0.414,0.605]","0.032 [0.022,0.043]","10265.147 [9678.693,10866.292]"
4,semantic_perplexity_15,"1261.687 [475.882,4628.324]","5538.000 [18.433,18335.338]","0.598 [0.502,0.678]","0.594 [0.498,0.674]","0.075 [0.055,0.096]","4589.509 [4298.794,4890.470]"
5,semantic_perplexity_5,"9250.801 [4809.699,29597.453]","186928.969 [32.228,827845.812]","0.375 [0.249,0.501]","0.454 [0.349,0.553]","0.004 [0.003,0.006]","187268.058 [177610.070,197607.723]"
6,syllables_per_min,"117.076 [28.693,251.182]","139.042 [32.973,777.020]","0.779 [0.720,0.839]","0.930 [0.884,0.967]","0.459 [0.384,0.545]","45.918 [37.484,55.169]"
7,first_person_percentage,"2.447 [0.447,5.763]","8.363 [1.160,24.390]","0.201 [0.088,0.323]","0.267 [0.155,0.379]","0.019 [0.008,0.031]","5.809 [5.537,6.102]"
8,first_person_sentiment_negative,"0.171 [0.042,0.749]","4.563 [0.275,10.142]","0.272 [0.161,0.383]","0.300 [0.191,0.404]","0.004 [0.002,0.006]","4.304 [4.127,4.479]"
9,first_person_sentiment_positive,"18.540 [10.902,31.237]","6.812 [2.470,39.785]","-0.197 [-0.297,-0.090]","-0.239 [-0.354,-0.121]","-0.042 [-0.067,-0.018]","-10.922 [-11.670,-10.178]"


In [2]:
# ---- B0: language-independent ----
B0_SUMMARY = [
    "file_length",
    "speech_length_minutes",
    "speech_length_words",
    "words_per_min",
    "speech_percentage",
    "mean_pre_word_pause",
    "mean_pause_variability",
    "word_repeat_percentage",
    "phrase_repeat_percentage",
    # optional session/turn aggregates (still language-independent)
    "num_turns",
    "num_one_word_turns",
    "mean_turn_length_minutes",
    "mean_turn_length_words",
    "mean_pre_turn_pause",
    "speaker_percentage",
    "num_interrupts",
]

stats_df = check_statistic(
    features=B0_SUMMARY,
    dataset_eng_dir="/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_eng_norm_gemma",
    dataset_ukr_dir="/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_ukr_norm_gemma",
    file_kind="summary",
    n_boot=2000,
    seed=1706,
)
stats_df.drop(columns=["note", 'n'], inplace=True)
stats_df

/var/folders/zz/9j6cqjd91k7190vj479mr1jw0000gn/T/ipykernel_40428/610093553.py:140: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.spearmanr(a, b, nan_policy="omit")


,feature,"EN_median[min,max]","UA_median[min,max]",r [95% CI],rho [95% CI],"ICC(2,1) [95% CI]",mean shift Δμ (UA-EN) [95% CI]
0,file_length,"14.912 [6.525,32.613]","14.912 [6.525,32.613]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
1,mean_pause_variability,"23.537 [2.971,40784.234]","23.537 [2.971,40784.234]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
2,mean_pre_word_pause,"4.240 [-25.161,10.041]","4.240 [-25.161,10.041]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
3,mean_turn_length_minutes,"14.563 [6.258,32.577]","14.563 [6.258,32.577]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
4,mean_turn_length_words,"88.000 [42.000,203.000]","88.000 [42.000,203.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
5,num_interrupts,"0.000 [0.000,0.000]","0.000 [0.000,0.000]","nan [nan,nan]","nan [nan,nan]","nan [nan,nan]","0.000 [0.000,0.000]"
6,num_one_word_turns,"0.000 [0.000,0.000]","0.000 [0.000,0.000]","nan [nan,nan]","nan [nan,nan]","nan [nan,nan]","0.000 [0.000,0.000]"
7,num_turns,"1.000 [1.000,1.000]","1.000 [1.000,1.000]","nan [nan,nan]","nan [nan,nan]","nan [nan,nan]","0.000 [0.000,0.000]"
8,phrase_repeat_percentage,"0.000 [0.000,8.333]","0.312 [0.000,8.333]","0.943 [0.892,0.978]","0.907 [0.860,0.949]","0.939 [0.882,0.977]","0.070 [0.035,0.106]"
9,speaker_percentage,"50.520 [12.622,177.834]","50.520 [12.622,177.834]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","-0.001 [-0.002,0.000]"


In [3]:
B0_TURNS = [
    "pre_turn_pause",
    "turn_length_minutes",
    "turn_length_words",
    "words_per_min",
    "speech_percentage",
    "mean_pause_length",
    "pause_variability",
    "word_repeat_percentage",
    "phrase_repeat_percentage",
    "interrupt_flag",
]
# 2) Turns files (turns_<pid>.csv), aggregated per participant by median:
stats_turns = check_statistic(
    features=B0_TURNS,  # your per-turn feature list
    dataset_eng_dir="/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_eng_norm_gemma",
    dataset_ukr_dir="/Users/pelmeshek1706/Desktop/projects/airest_notebooks/result_oppenwillis_ukr_norm_gemma",
    file_kind="turns",
    turns_agg="median",
    # participant_only=True,
)

stats_turns.drop(columns=["note", 'n'], inplace=True)
stats_turns

/var/folders/zz/9j6cqjd91k7190vj479mr1jw0000gn/T/ipykernel_40428/610093553.py:140: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, _ = stats.spearmanr(a, b, nan_policy="omit")


,feature,"EN_median[min,max]","UA_median[min,max]",r [95% CI],rho [95% CI],"ICC(2,1) [95% CI]",mean shift Δμ (UA-EN) [95% CI]
0,interrupt_flag,"0.000 [0.000,0.000]","0.000 [0.000,0.000]","nan [nan,nan]","nan [nan,nan]","nan [nan,nan]","0.000 [0.000,0.000]"
1,mean_pause_length,"4.240 [-25.161,10.041]","4.240 [-25.161,10.041]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
2,pause_variability,"23.257 [2.898,39897.620]","23.257 [2.898,39897.620]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
3,phrase_repeat_percentage,"0.000 [0.000,8.333]","0.312 [0.000,8.333]","0.943 [0.892,0.978]","0.907 [0.860,0.949]","0.939 [0.882,0.977]","0.070 [0.035,0.106]"
4,speech_percentage,"57.263 [12.707,259.993]","57.263 [12.707,259.993]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
5,turn_length_minutes,"14.563 [6.258,32.577]","14.563 [6.258,32.577]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
6,turn_length_words,"88.000 [42.000,203.000]","88.000 [42.000,203.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
7,word_repeat_percentage,"0.897 [0.000,20.889]","1.169 [0.000,20.889]","0.951 [0.903,0.982]","0.930 [0.896,0.957]","0.939 [0.872,0.978]","0.243 [0.178,0.317]"
8,words_per_min,"6.394 [2.044,10.662]","6.394 [2.044,10.662]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","1.000 [1.000,1.000]","0.000 [0.000,0.000]"
9,pre_turn_pause,"nan [nan,nan]","nan [nan,nan]","nan [nan,nan]","nan [nan,nan]","nan [nan,nan]","nan [nan,nan]"
